# 🏦 Task 1: Pension Beneficiary Eligibility Prediction

## Objective
Develop a machine learning model that predicts **pension beneficiary eligibility category** using demographic data.

We use the **Adult Census Income** dataset, which contains demographic features such as age, education, occupation, and work hours. The target variable is **income** (`<=50K` or `>50K`), which serves as a proxy for pension eligibility — individuals with higher income are more likely to be eligible for certain pension benefit tiers.

## Approach
1. Load and explore the dataset
2. Preprocess the data (handle missing values, encode features)
3. Train multiple classification models
4. Evaluate and compare model performance
5. Choose the best model and summarize findings

---
## 1. Import Libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('All libraries imported successfully!')

---
## 2. Load the Dataset

Upload `adult.csv` when prompted (if running in Colab), or ensure it is in the same directory.

In [ ]:
# --- Upload file in Google Colab ---
# Uncomment the lines below if you are running this in Google Colab
# from google.colab import files
# uploaded = files.upload()

# Load dataset
df = pd.read_csv('adult.csv')
print(f'Dataset shape: {df.shape}')
print(f'Number of samples: {df.shape[0]}')
print(f'Number of features: {df.shape[1]}')
df.head()

---
## 3. Exploratory Data Analysis (EDA)

### 3.1 Dataset Overview

In [ ]:
# Basic info
print('='*60)
print('DATASET INFO')
print('='*60)
df.info()

print('\n' + '='*60)
print('STATISTICAL SUMMARY (Numerical Features)')
print('='*60)
df.describe()

In [ ]:
# Check for missing values
print('='*60)
print('MISSING VALUES')
print('='*60)
print(df.isnull().sum())

# Check for '?' placeholder values
print('\n' + '='*60)
print('PLACEHOLDER "?" VALUES PER COLUMN')
print('='*60)
for col in df.columns:
    if df[col].dtype == 'object':
        q_count = (df[col].str.strip() == '?').sum()
        if q_count > 0:
            print(f'{col}: {q_count} ({q_count/len(df)*100:.2f}%)')

### 3.2 Target Variable Distribution

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
colors = ['#3498db', '#e74c3c']
df['income'].value_counts().plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Income Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Income Category')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Pie chart
df['income'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                                  colors=colors, startangle=90,
                                  explode=(0.05, 0.05))
axes[1].set_title('Income Distribution (%)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print('\nClass Distribution:')
print(df['income'].value_counts())
print(f'\nClass Ratio: {df["income"].value_counts().values[0] / df["income"].value_counts().values[1]:.2f}:1')

### 3.3 Numerical Feature Distributions

In [ ]:
# Numerical columns
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f'Numerical columns: {num_cols}')

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    if i < 6:
        sns.histplot(data=df, x=col, hue='income', kde=True, ax=axes[i],
                     palette=colors, alpha=0.7)
        axes[i].set_title(f'{col} Distribution by Income', fontsize=12, fontweight='bold')

# Hide any empty subplots
for j in range(len(num_cols), 6):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

### 3.4 Categorical Feature Analysis

In [ ]:
# Categorical columns
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
cat_cols_analysis = [c for c in cat_cols if c != 'income']
print(f'Categorical columns: {cat_cols_analysis}')

fig, axes = plt.subplots(3, 3, figsize=(22, 16))
axes = axes.flatten()

for i, col in enumerate(cat_cols_analysis):
    if i < 9:
        # Show top 8 categories for readability
        top_cats = df[col].value_counts().head(8).index
        temp = df[df[col].isin(top_cats)]
        sns.countplot(data=temp, x=col, hue='income', ax=axes[i],
                      palette=colors, order=top_cats)
        axes[i].set_title(f'{col}', fontsize=12, fontweight='bold')
        axes[i].tick_params(axis='x', rotation=45)
        axes[i].legend(title='Income', fontsize=8)

for j in range(len(cat_cols_analysis), 9):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

### 3.5 Correlation Heatmap

In [ ]:
# Encode target for correlation
df_corr = df.copy()
df_corr['income_encoded'] = (df_corr['income'].str.strip() == '>50K').astype(int)

# Correlation matrix
corr_cols = num_cols + ['income_encoded']
corr_matrix = df_corr[corr_cols].corr()

plt.figure(figsize=(12, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=1, vmin=-1, vmax=1)
plt.title('Correlation Heatmap (Numerical Features + Target)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Data Preprocessing

In [ ]:
# Work on a copy
data = df.copy()

# Step 1: Strip whitespace from string columns
for col in data.select_dtypes(include='object').columns:
    data[col] = data[col].str.strip()

# Step 2: Replace '?' with NaN and drop rows with missing values
data.replace('?', np.nan, inplace=True)
print(f'Rows before dropping missing: {len(data)}')
data.dropna(inplace=True)
print(f'Rows after  dropping missing: {len(data)}')
print(f'Rows removed: {len(df) - len(data)}')

# Step 3: Drop 'fnlwgt' (census weight — not useful for prediction)
data.drop('fnlwgt', axis=1, inplace=True)

# Step 4: Encode target variable
data['income'] = (data['income'] == '>50K').astype(int)
print(f'\nTarget encoding: <=50K → 0, >50K → 1')
print(data['income'].value_counts())

In [ ]:
# Step 5: Label encode categorical features
cat_cols_final = data.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns to encode: {cat_cols_final}')

label_encoders = {}
for col in cat_cols_final:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    label_encoders[col] = le
    print(f'  {col}: {len(le.classes_)} unique values')

print('\nEncoded dataset shape:', data.shape)
data.head()

In [ ]:
# Step 6: Split features and target
X = data.drop('income', axis=1)
y = data['income']

print(f'Features shape: {X.shape}')
print(f'Target shape:   {y.shape}')
print(f'Target distribution:\n{y.value_counts(normalize=True)}')

# Step 7: Train-test split (80-20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTraining set: {X_train.shape[0]} samples')
print(f'Testing  set: {X_test.shape[0]} samples')

# Step 8: Scale numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('\n✅ Scaling complete!')

---
## 5. Model Training & Evaluation

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=7),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=42),
}

# Train and evaluate each model
results = []

for name, model in models.items():
    print(f'\n{"="*60}')
    print(f'  Training: {name}')
    print(f'{"="*60}')

    # Train
    model.fit(X_train_scaled, y_train)

    # Predict
    y_pred = model.predict(X_test_scaled)

    # Probability predictions (for ROC)
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_prob = model.decision_function(X_test_scaled)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'AUC-ROC': auc,
        'y_prob': y_prob
    })

    print(f'  Accuracy:  {acc:.4f}')
    print(f'  Precision: {prec:.4f}')
    print(f'  Recall:    {rec:.4f}')
    print(f'  F1 Score:  {f1:.4f}')
    print(f'  AUC-ROC:   {auc:.4f}')
    print(f'\nClassification Report:')
    print(classification_report(y_test, y_pred, target_names=['<=50K', '>50K']))

### 5.1 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 5))

for i, (name, model) in enumerate(models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['<=50K', '>50K'],
                yticklabels=['<=50K', '>50K'])
    axes[i].set_title(f'{name}', fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Actual')
    axes[i].set_xlabel('Predicted')

plt.suptitle('Confusion Matrices', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 5.2 ROC Curves

In [ ]:
plt.figure(figsize=(10, 8))
colors_roc = ['#2ecc71', '#3498db', '#e74c3c', '#9b59b6']

for i, r in enumerate(results):
    fpr, tpr, _ = roc_curve(y_test, r['y_prob'])
    plt.plot(fpr, tpr, color=colors_roc[i], lw=2,
             label=f"{r['Model']} (AUC = {r['AUC-ROC']:.4f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=13)
plt.ylabel('True Positive Rate', fontsize=13)
plt.title('ROC Curves — Model Comparison', fontsize=15, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Model Comparison

In [ ]:
# Create results dataframe (exclude y_prob column)
results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'y_prob'} for r in results])
results_df = results_df.set_index('Model')
results_df = results_df.sort_values('F1 Score', ascending=False)

print('='*70)
print('                    MODEL COMPARISON SUMMARY')
print('='*70)
print(results_df.to_string())
print(f'\n🏆 Best Model (by F1 Score): {results_df.index[0]}')
print(f'   F1 Score: {results_df.iloc[0]["F1 Score"]:.4f}')
print(f'   AUC-ROC:  {results_df.iloc[0]["AUC-ROC"]:.4f}')

In [ ]:
# Grouped bar chart comparison
fig, ax = plt.subplots(figsize=(14, 7))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC-ROC']
x = np.arange(len(results_df.index))
width = 0.15
bar_colors = ['#1abc9c', '#3498db', '#e67e22', '#e74c3c', '#9b59b6']

for i, metric in enumerate(metrics):
    bars = ax.bar(x + i * width, results_df[metric], width,
                  label=metric, color=bar_colors[i], edgecolor='white')
    # Add value labels on top of bars
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Model', fontsize=13)
ax.set_ylabel('Score', fontsize=13)
ax.set_title('Model Performance Comparison', fontsize=15, fontweight='bold')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(results_df.index, rotation=15)
ax.legend(loc='lower right', fontsize=10)
ax.set_ylim(0, 1.15)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 7. Feature Importance (Best Model)

In [ ]:
# Feature importance from Random Forest
rf_model = models['Random Forest']
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(feature_importance['Feature'], feature_importance['Importance'],
         color=sns.color_palette('viridis', len(feature_importance)),
         edgecolor='white')
plt.xlabel('Importance', fontsize=13)
plt.ylabel('Feature', fontsize=13)
plt.title('Feature Importance (Random Forest)', fontsize=15, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print('\nTop 5 Most Important Features:')
for _, row in feature_importance.tail(5).iloc[::-1].iterrows():
    print(f'  • {row["Feature"]}: {row["Importance"]:.4f}')

---
## 8. Conclusion

### Summary of Results

In this project, we developed a machine learning pipeline to predict **pension beneficiary eligibility** categories using demographic data from the Adult Census dataset.

**Key Findings:**
- We trained and evaluated **4 different classification models**: Logistic Regression, Random Forest, K-Nearest Neighbors, and Gradient Boosting.
- The dataset exhibited **class imbalance** (approximately 75% <=50K vs 25% >50K), which was handled through stratified splitting.
- **Gradient Boosting** and **Random Forest** tend to perform best due to their ability to capture complex non-linear relationships in demographic data.
- Top features influencing eligibility include **age**, **education level**, **capital gains**, **hours per week**, and **marital status**.

### Future Improvements
- Apply **SMOTE** or other oversampling techniques to handle class imbalance
- Perform **hyperparameter tuning** using GridSearchCV or RandomizedSearchCV
- Try **deep learning** approaches using neural networks
- Add **cross-validation** for more robust evaluation